In [1]:
import pandas as pd

In [3]:
df_2001 = pd.read_excel(
    '../../raw/Occupational Employment and Wage Statistics/Wage_2001.xls',
    engine='xlrd'
)
df_2001.head()

,occ_code,occ_title,group,tot_emp,emp_prse,h_mean,a_mean,mean_prse,h_wpct10,h_wpct25,h_median,h_wpct75,h_wpct90,a_wpct10,a_wpct25,a_median,a_wpct75,a_wpct90,annual,year
0,00-0000,All Occupations,NaN,127980410,0.1,16.35,34020,0.2,6.76,8.72,13.01,20.31,30.23,14070,18140,27060,42240,62880,NaN,2001
1,11-0000,Management Occupations,major,7212360,0.3,34.04,70800,0.2,14.62,20.84,30.88,44.77,66.62,30410,43340,64220,93120,138570,NaN,2001
2,11-1011,Chief Executives,NaN,455930,0.8,51.77,107670,0.4,23.52,36.93,57.91,#,#,48920,76810,120450,#,#,NaN,2001
3,11-1021,General and Operations Managers,NaN,2064220,0.4,35.37,73570,0.2,14.99,20.81,31.25,48.22,#,31190,43280,65000,100300,#,NaN,2001
4,11-1031,Legislators,NaN,67400,2.8,13.54,28170,1.8,5.69,6.17,7.05,17,31.2,11830,12840,14650,35360,64890,NaN,2001


In [4]:
occ_codes = ['15-1011', '15-1021', '15-1031', '15-1032', '15-1041', '15-1051', '15-1061', '15-1071', '15-1081', '15-1099']

df_filtered = df_2001[df_2001['occ_code'].isin(occ_codes)][['occ_code', 'occ_title', 'tot_emp', 'a_mean']]
print(df_filtered)

   occ_code                                         occ_title  tot_emp a_mean
59  15-1011     Computer and Information Scientists, Research    25620  76970
60  15-1021                              Computer Programmers   501550  62890
61  15-1031         Computer Software Engineers, Applications   361690  72370
62  15-1032     Computer Software Engineers, Systems Software   261520  74490
63  15-1041                      Computer Support Specialists   493240  41920
64  15-1051                         Computer Systems Analysts   448270  63710
65  15-1061                           Database Administrators   104250  58420
66  15-1071       Network and Computer Systems Administrators   227840  56440
67  15-1081  Network Systems and Data Communications Analysts   126060  60300


In [13]:
# Load and filter IT occupations for years 2002-2024
# Note: BLS changed occupation codes in 2010, so we use title keywords for 2010+
all_years_data = []

def get_engine(file_path):
    return 'openpyxl' if file_path.endswith('.xlsx') else 'xlrd'

years = list(range(2002, 2025))

# IT job title keywords for 2010+
it_keywords = ['computer', 'programmer', 'software engineer', 'database', 'network', 'systems analyst']

for year in years:
    try:
        file_path = f'../../raw/Occupational Employment and Wage Statistics/Wage_{year}.xls'
        if year >= 2014:
            file_path = file_path.replace('.xls', '.xlsx')
        
        df_year = pd.read_excel(file_path, engine=get_engine(file_path))
        df_year.columns = df_year.columns.str.lower()
        
        if year <= 2009:
            # Use exact occupation codes for 2002-2009
            df_filtered_year = df_year[df_year['occ_code'].isin(occ_codes)][['occ_code', 'occ_title', 'tot_emp', 'a_mean']]
        else:
            # Use title keyword search for 2010+
            mask = df_year['occ_title'].str.lower().str.contains('|'.join(it_keywords), na=False)
            df_filtered_year = df_year[mask][['occ_code', 'occ_title', 'tot_emp', 'a_mean']]
            # Filter out very broad categories
            df_filtered_year = df_filtered_year[~df_filtered_year['occ_title'].str.contains('all occupations', case=False, na=False)]
        
        if len(df_filtered_year) > 0:
            # Convert numeric columns to numeric type
            df_filtered_year['tot_emp'] = pd.to_numeric(df_filtered_year['tot_emp'], errors='coerce')
            df_filtered_year['a_mean'] = pd.to_numeric(df_filtered_year['a_mean'], errors='coerce')
            
            df_filtered_year['year'] = year
            all_years_data.append(df_filtered_year)
            print(f"✓ Loaded {year} - {len(df_filtered_year)} rows")
        else:
            print(f"○ Loaded {year} - 0 rows")
    except Exception as e:
        print(f"✗ Error loading {year}: {str(e)[:50]}")

if all_years_data:
    df_all_years = pd.concat(all_years_data, ignore_index=True)
    print(f"\nTotal records: {len(df_all_years)}")
    print(f"Years covered: {sorted(df_all_years['year'].unique())}")
    print("\nRecords by year:")
    print(df_all_years.groupby('year').size())
else:
    print("No data loaded")

✓ Loaded 2002 - 9 rows
✓ Loaded 2003 - 9 rows
✓ Loaded 2004 - 10 rows
✓ Loaded 2005 - 10 rows
✓ Loaded 2006 - 10 rows
✓ Loaded 2007 - 10 rows
✓ Loaded 2008 - 10 rows
✓ Loaded 2009 - 10 rows
✓ Loaded 2010 - 18 rows
✓ Loaded 2011 - 18 rows
✓ Loaded 2012 - 33 rows
✓ Loaded 2013 - 33 rows
✓ Loaded 2014 - 33 rows
✓ Loaded 2015 - 33 rows
✓ Loaded 2016 - 33 rows
✓ Loaded 2017 - 33 rows
✓ Loaded 2018 - 33 rows
✓ Loaded 2019 - 31 rows
✓ Loaded 2020 - 31 rows
✓ Loaded 2021 - 32 rows
✓ Loaded 2022 - 32 rows
✓ Loaded 2023 - 32 rows
✓ Loaded 2024 - 32 rows

Total records: 535
Years covered: [np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Records by year:
year
2002     9
2003     9
2

In [10]:
# Quick check: do our codes exist in 2010?
df_2010 = pd.read_excel('../../raw/Occupational Employment and Wage Statistics/Wage_2010.xls', engine='xlrd')
df_2010.columns = df_2010.columns.str.lower()

codes_found = df_2010[df_2010['occ_code'].isin(occ_codes)]
print(f"Found {len(codes_found)} matching codes in 2010")
print("\nDifference between 2001 and 2010:")
print(f"2001 unique codes: {df_2001['occ_code'].nunique()}")
print(f"2010 unique codes: {df_2010['occ_code'].nunique()}")
print(f"Our target codes that exist in 2001: {len(df_2001[df_2001['occ_code'].isin(occ_codes)])}")
print(f"Our target codes that exist in 2010: {len(codes_found)}")

Found 0 matching codes in 2010

Difference between 2001 and 2010:
2001 unique codes: 733
2010 unique codes: 819
Our target codes that exist in 2001: 9
Our target codes that exist in 2010: 0


In [14]:
# Summary and sample of complete dataset
print("Complete IT Occupations Dataset (2002-2024)")
print("=" * 50)
print(f"\nTotal rows: {len(df_all_years)}")
print(f"Unique years: {df_all_years['year'].nunique()}")
print(f"Unique occupations: {df_all_years['occ_code'].nunique()}")

print("\n--- Sample Data ---")
print(df_all_years.head(10))

print("\n--- Year Summary ---")
yearly_stats = df_all_years.groupby('year').agg({
    'occ_code': 'count',
    'tot_emp': 'sum',
    'a_mean': 'mean'
}).round(0)
yearly_stats.columns = ['count', 'total_employment', 'avg_annual_wage']
print(yearly_stats)

# Optional: Save to CSV
# df_all_years.to_csv('IT_occupations_2002_2024.csv', index=False)
# print("\n✓ Saved to IT_occupations_2002_2024.csv")

Complete IT Occupations Dataset (2002-2024)

Total rows: 535
Unique years: 23
Unique occupations: 66

--- Sample Data ---
  occ_code                                         occ_title  tot_emp  a_mean  \
0  15-1011     Computer and information scientists, research    23210   84530   
1  15-1021                              Computer programmers   431640   64510   
2  15-1031         Computer software engineers, applications   392140   75750   
3  15-1032     Computer software engineers, systems software   285760   78400   
4  15-1041                      Computer support specialists   482990   42640   
5  15-1051                         Computer systems analysts   474780   66180   
6  15-1061                           Database administrators   100890   61440   
7  15-1071       Network and computer systems administrators   237980   59140   
8  15-1081  Network systems and data communications analysts   148030   62060   
9  15-1011     Computer and information scientists, research    2377

In [15]:
from pathlib import Path

# Export one CSV per year (2001-2024) with only CS/IT-related jobs and required columns
input_dir = Path('../../raw/Occupational Employment and Wage Statistics')
output_dir = Path('.')
output_dir.mkdir(parents=True, exist_ok=True)

# Legacy SOC codes available in early years
legacy_occ_codes = {
    '15-1011', '15-1021', '15-1031', '15-1032', '15-1041',
    '15-1051', '15-1061', '15-1071', '15-1081', '15-1099'
}

# CS/IT title keywords used for newer years where code mapping changed
it_title_pattern = (
    r'computer|software|programmer|web developer|database|network|'
    r'information security|systems analyst|data scientist|'
    r'computer support|computer systems|cloud'
)

written_files = []
for file in sorted(input_dir.glob('Wage_*.*')):
    year = int(file.stem.split('_')[-1])

    engine = 'openpyxl' if file.suffix.lower() == '.xlsx' else 'xlrd'
    df = pd.read_excel(file, engine=engine)
    df.columns = df.columns.str.lower()

    # Normalize key columns and keep only what we need.
    if 'occ_code' not in df.columns or 'occ_title' not in df.columns:
        print(f'Skipped {year}: missing occ_code/occ_title')
        continue

    if year <= 2009:
        mask = df['occ_code'].astype(str).str.strip().isin(legacy_occ_codes)
    else:
        mask = df['occ_title'].astype(str).str.lower().str.contains(it_title_pattern, regex=True, na=False)

    out = df.loc[mask, ['occ_code', 'occ_title', 'tot_emp', 'a_mean']].copy()
    out['tot_emp'] = pd.to_numeric(out['tot_emp'], errors='coerce')
    out['a_mean'] = pd.to_numeric(out['a_mean'], errors='coerce')

    out_path = output_dir / f'IT_jobs_{year}.csv'
    out.to_csv(out_path, index=False)
    written_files.append(out_path.name)
    print(f'Wrote {out_path.name}: {len(out)} rows')

print(f'\nDone. CSV files created: {len(written_files)}')

Wrote IT_jobs_2001.csv: 9 rows
Wrote IT_jobs_2002.csv: 9 rows
Wrote IT_jobs_2003.csv: 9 rows
Wrote IT_jobs_2004.csv: 10 rows
Wrote IT_jobs_2005.csv: 10 rows
Wrote IT_jobs_2006.csv: 10 rows
Wrote IT_jobs_2007.csv: 10 rows
Wrote IT_jobs_2008.csv: 10 rows
Wrote IT_jobs_2009.csv: 10 rows
Wrote IT_jobs_2010.csv: 20 rows
Wrote IT_jobs_2011.csv: 20 rows
Wrote IT_jobs_2012.csv: 37 rows
Wrote IT_jobs_2013.csv: 37 rows
Wrote IT_jobs_2014.csv: 37 rows
Wrote IT_jobs_2015.csv: 37 rows
Wrote IT_jobs_2016.csv: 37 rows
Wrote IT_jobs_2017.csv: 37 rows
Wrote IT_jobs_2018.csv: 37 rows
Wrote IT_jobs_2019.csv: 35 rows
Wrote IT_jobs_2020.csv: 35 rows
Wrote IT_jobs_2021.csv: 38 rows
Wrote IT_jobs_2022.csv: 38 rows
Wrote IT_jobs_2023.csv: 38 rows
Wrote IT_jobs_2024.csv: 38 rows

Done. CSV files created: 24


In [16]:
# Build yearly summary: average CS/IT salary and total employment per year
# Include 2001 data (already in df_filtered) alongside df_all_years (2002-2024)
df_2001_it = df_2001[df_2001['occ_code'].isin(legacy_occ_codes)][['occ_code', 'tot_emp', 'a_mean']].copy()
df_2001_it['tot_emp'] = pd.to_numeric(df_2001_it['tot_emp'], errors='coerce')
df_2001_it['a_mean']  = pd.to_numeric(df_2001_it['a_mean'],  errors='coerce')
df_2001_it['year'] = 2001

combined = pd.concat([df_2001_it[['year', 'tot_emp', 'a_mean']],
                       df_all_years[['year', 'tot_emp', 'a_mean']]], ignore_index=True)

yearly_summary = (
    combined.groupby('year', as_index=False)
    .agg(total_employment=('tot_emp', 'sum'),
         avg_annual_salary=('a_mean', 'mean'))
)
yearly_summary['avg_annual_salary'] = yearly_summary['avg_annual_salary'].round(0).astype(int)
yearly_summary['total_employment']  = yearly_summary['total_employment'].astype('Int64')

out_path = Path('.') / 'IT_yearly_summary.csv'
yearly_summary.to_csv(out_path, index=False)

print(f"Saved: {out_path.resolve()}")
print(f"\n{yearly_summary.to_string(index=False)}")

Saved: D:\projects\Customer-Churn-Prediction\Data\Integrated\Occupational Employment and Wage Statistics\IT_yearly_summary.csv

 year  total_employment  avg_annual_salary
 2001           2550040              63057
 2002           2577420              66072
 2003           2594750              66784
 2004           2814290              67678
 2005           2855320              69605
 2006           2969510              72132
 2007           3083130              75146
 2008           3198050              77429
 2009           3196930              79443
 2010           6502900              71093
 2011           6719940              72211
 2012          14659350              73388
 2013          15080540              75093
 2014          15532340              76890
 2015          16089740              79034
 2016          16584180              80975
 2017          16915240              82829
 2018          17355520              84480
 2019          17837530              89113
 2020       